<a href="https://colab.research.google.com/github/iilnreddy/ljmu/blob/main/Models/Benchmarking_USA_OHLCV_Prediction_return.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Returns vs Log Returns as target

# Benchmarking using Log Returns

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# =========================
# FEATURE ENGINEERING
# =========================
def create_features(df):
    df = df.copy()

    df["ret_1"] = df["Close"].pct_change()
    df["ret_5"] = df["Close"].pct_change(5)

    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_10"] = df["Close"].rolling(10).mean()

    df["ma_ratio_5"] = df["Close"] / df["ma_5"]

    df["volatility_5"] = df["Close"].rolling(5).std()
    df["volatility_10"] = df["Close"].rolling(10).std()

    df["vol_ratio"] = df["Volume"] / df["Volume"].rolling(10).mean()

    df["hl_spread"] = (df["High"] - df["Low"]) / df["Close"]

    # 🎯 Target: LOG RETURN
    df["target"] = np.log(df["Close"].shift(-1) / df["Close"])

    df.dropna(inplace=True)

    return df


# =========================
# TRAIN-TEST SPLIT
# =========================
def split_data(df):
    split = int(len(df) * 0.8)
    return df.iloc[:split], df.iloc[split:]


# =========================
# EVALUATION METRICS
# =========================
def evaluate(y_true, y_pred):
    df_eval = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})

    # Convert log → simple return
    y_pred_simple = np.exp(y_pred) - 1
    y_true_simple = np.exp(y_true) - 1

    # Direction accuracy
    direction_acc = np.mean(np.sign(y_true_simple) == np.sign(y_pred_simple))

    # Strategy returns
    df_eval["strategy"] = y_pred_simple * y_true_simple
    sharpe = df_eval["strategy"].mean() / df_eval["strategy"].std() * np.sqrt(252)

    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "Direction_Accuracy": direction_acc,
        "Sharpe": sharpe
    }


# =========================
# MODEL DEFINITIONS
# =========================
def get_models():
    return {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=100),
        "XGBoost": XGBRegressor(n_estimators=100, max_depth=5, verbosity=0),
        "LightGBM": LGBMRegressor(n_estimators=100)
    }


# =========================
# RUN BENCHMARK
# =========================
def run_model_comparison(df):

    df = create_features(df)

    #features = [col for col in df.columns if col not in ["Date", "target"]]
    features = df.select_dtypes(include=[np.number]).columns.tolist()
    features.remove("target")

    train, test = split_data(df)

    X_train = train[features]
    y_train = train["target"]

    X_test = test[features]
    y_test = test["target"]

    models = get_models()
    results = []

    for name, model in models.items():
        print(f"Training {name}...")

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        metrics = evaluate(y_test, preds)

        metrics["Model"] = name
        results.append(metrics)

    results_df = pd.DataFrame(results)

    # Rank by Sharpe
    results_df.sort_values(by="Sharpe", ascending=False, inplace=True)

    return results_df


In [ ]:
from pathlib import Path
RAW_DIR = Path("../Data/normalized_csv")
df= pd.read_csv(RAW_DIR / "USA_OHLCV_STOCKS.csv")

In [ ]:
df.head()

,Date,Open,High,Low,Close,Volume,Stock Symbol,Stock Name,Stock Market,INDEX
0,2000-01-03,46.963906,47.075726,40.180231,42.938431,4674353.0,A,Agilent Technologies,NYSE/NASDAQ,['S&P 500']
1,2000-01-04,40.627500,41.074778,38.614762,39.658405,4765083.0,A,Agilent Technologies,NYSE/NASDAQ,['S&P 500']
2,2000-01-05,39.509329,39.658423,35.968401,37.198406,5758642.0,A,Agilent Technologies,NYSE/NASDAQ,['S&P 500']
3,2000-01-06,36.751126,36.974765,34.663840,35.782028,2534434.0,A,Agilent Technologies,NYSE/NASDAQ,['S&P 500']
4,2000-01-07,35.222933,39.322957,35.185662,38.763863,2819626.0,A,Agilent Technologies,NYSE/NASDAQ,['S&P 500']


In [ ]:


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)

    results = run_model_comparison(df)

    print("\n📊 Model Comparison:")
    print(results)

C:\Users\iilnr\AppData\Local\Temp\ipykernel_53780\1308981915.py:18: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ret_1"] = df["Close"].pct_change()
C:\Users\iilnr\AppData\Local\Temp\ipykernel_53780\1308981915.py:19: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ret_5"] = df["Close"].pct_change(5)


Training LinearRegression...
Training RandomForest...
Training GradientBoosting...


MemoryError: Unable to allocate 4.55 MiB for an array with shape (596470,) and data type float64

# OpenAI - ChatGPT

This is a very informative result — let’s interpret it like a quant would, step by step 👇

📊 Your Results
Model	MSE	Direction Accuracy	Sharpe
LightGBM	0.000395	53.25%	0.98 🔥
XGBoost	0.000654	49.24%	0.41
Linear Regression	0.000489	51.08%	0.14
Random Forest	0.000622	47.40%	-0.25 ❌
Gradient Boosting	0.001931	47.18%	-0.29 ❌
🧠 Step 1: What Each Metric Means (Quick Recap)
MSE → prediction error (lower is better)
Direction Accuracy → % of correct up/down predictions
Sharpe Ratio → risk-adjusted trading performance (MOST IMPORTANT)
🔥 Step 2: The Most Important Insight

👉 LightGBM is clearly the best model

Not just slightly — by a BIG margin

🧠 Step 3: Deep Interpretation
🥇 1. LightGBM (Clear Winner)
Sharpe: 0.98  ← VERY strong
Accuracy: 53% ← solid edge
MSE: low
🔍 What this means:
Model is consistently profitable
Predictions are:
reasonably accurate
well-calibrated
Produces stable signals (low noise)

👉 This is exactly what you want in trading

🥈 2. XGBoost (Decent but weaker)
Sharpe: 0.40
Accuracy: < 50% (bad)
🔍 Interesting observation:
Accuracy is below 50% ❗
Yet Sharpe is positive

👉 This means:

Model makes fewer but better-quality bets

🧠 IMPORTANT INSIGHT

You don’t need >50% accuracy to make money

If:

Wins are bigger than losses
Model avoids bad trades

👉 You can still profit

🥉 3. Linear Regression (Baseline)
Sharpe: 0.14 (weak)
Accuracy: ~51%
Interpretation:
Slight edge
But too weak to trade seriously

👉 Good as a baseline only

❌ 4. Random Forest (Losing Model)
Sharpe: -0.25
Accuracy: 47%

👉 Model is:

Wrong more often than right
Produces bad signals
❌ 5. Gradient Boosting (Worst)
Sharpe: -0.29
MSE: highest

👉 Completely unusable for trading

⚠️ Step 4: Why MSE is MISLEADING

Look:

Model	MSE	Sharpe
LightGBM	0.000395	0.98 ✅
Linear	0.000489	0.14 ❌

👉 MSE difference is small
👉 Sharpe difference is HUGE

🔥 CRITICAL LESSON

Low error ≠ profitable model

🧠 Step 5: What Actually Matters
✅ Sharpe Ratio drives decisions
Sharpe	Meaning
< 0	losing strategy ❌
0–0.5	weak
0.5–1	good
~1	very strong 🔥

👉 Your LightGBM ≈ 1 → excellent

📉 Step 6: Why Some Models Fail
Random Forest / Gradient Boosting

Likely issues:

Overfitting
Noisy predictions
Poor generalization
LightGBM Advantage

LightGBM is better because:

Handles tabular data extremely well
Captures non-linear patterns efficiently
Regularization works better
🚀 Step 7: Final Ranking (What You Should Do)
✅ Use:
1. LightGBM  ← PRIMARY MODEL 🔥
2. XGBoost   ← backup / ensemble
❌ Avoid:
RandomForest
GradientBoosting
🧠 Final Takeaway (Very Important)

Your experiment shows a real quant result:

Accuracy is not everything
MSE is not important
👉 Sharpe Ratio decides the winner
🚀 What to do next (this is where edge comes)

I recommend:

1. Improve LightGBM
Tune hyperparameters
Add more features (RSI, Bollinger Bands)
2. Build Ensemble
Final Prediction = LightGBM + XGBoost
3. Add Risk Filters
Trade only when confidence is high